Roshan Gopal Trading Project - 2026

Cross Market Momentum for Basketball on Kalshi

Research Question
    Does time series momentum exist between individual game markets (Team vs Team B) and series markets (Will Team A or Team B win the series) during the NBA Playoffs?

Strategy Design

Universe: NBA games from the 2026 Eastern Conference Finals and the 2026 Western Conference Finals and their respective series markets

Data: Prices of "Yes" and "No" throughout the duration of the game and series from the Kalshi API v2 candlestick data

Strategy: Extract momentum signal from changes in game prices and use them to predict--and trade on--series prices.  

Hypothesis: Momentum between games and a series exists at a rolling mean over 3 minutes with a threshold of 0.01. 

Process: Utilized Grid Search over varying rolling means and thresholds. Tested strategy on unseen data in order to validate its potency. 

Execution: Dollar nuetral weighting on all signals above the threshold. Live trader places limit orders in order to avoid heavy fees. 

Results: 2.25 Sharpe Ratio on training set and 2.0 Sharpe Ratio on unseen test data. Average return of 30% over 4 games from the Western Conference Finals and the NBA Finals.

Limitations: Historical data from Kalshi only appears every minute so the strategy might miss intra-minute changes. 

Part 1: Install Dependencies, pull data via the Kalshi Api, and define some useful functions

In [ ]:
import requests
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import scipy.stats as stats
from datetime import datetime, timedelta, timezone

KALSHI_BASE = "https://external-api.kalshi.com/trade-api/v2"

PLAYOFF_SERIES = {
    "ECF_NYK_CLE": {
        "teams": {
            "Knicks":    "KXNBAEAST-26-NYK",
            "Cavaliers": "KXNBAEAST-26-CLE",
        },
        "game_tickers": {
            "2026-05-19": ("KXNBAGAME-26MAY19CLENYK-NYK", datetime(2026, 5, 20, 0, 0, tzinfo=timezone.utc)),
            "2026-05-21": ("KXNBAGAME-26MAY21CLENYK-NYK", datetime(2026, 5, 22, 0, 0, tzinfo=timezone.utc)),
            "2026-05-23": ("KXNBAGAME-26MAY23NYKCLE-CLE", datetime(2026, 5, 24, 0, 0, tzinfo=timezone.utc)),
            "2026-05-25": ("KXNBAGAME-26MAY25NYKCLE-CLE", datetime(2026, 5, 26, 0, 0, tzinfo=timezone.utc)),
        }
    },
    "WCF_OKC_SAS": {
        "teams": {
            "Thunder": "KXNBAWEST-26-OKC",
            "Spurs":   "KXNBAWEST-26-SAS",
        },
        "game_tickers": {
            "2026-05-18": ("KXNBAGAME-26MAY18SASOKC-OKC", datetime(2026, 5, 19, 0, 30, tzinfo=timezone.utc)),
            "2026-05-20": ("KXNBAGAME-26MAY20SASOKC-OKC", datetime(2026, 5, 21, 0, 30, tzinfo=timezone.utc)),
            "2026-05-22": ("KXNBAGAME-26MAY22OKCSAS-SAS", datetime(2026, 5, 23, 0, 30, tzinfo=timezone.utc)),
            "2026-05-24": ("KXNBAGAME-26MAY24OKCSAS-SAS", datetime(2026, 5, 25, 0, 30, tzinfo=timezone.utc)),
            "2026-05-26": ("KXNBAGAME-26MAY26SASOKC-OKC", datetime(2026, 5, 27, 0, 30, tzinfo=timezone.utc)),
            
        }
    }
}

In [ ]:
# Search for all KXNBAGAME tickers with MAY28 or MAY30
resp = requests.get(f"{KALSHI_BASE}/markets", 
                    params={"series_ticker": "KXNBAGAME", "limit": 100})
markets = resp.json().get("markets", [])
for m in markets:
    if "MAY28" in m["ticker"] or "MAY30" in m["ticker"]:
        print(m["ticker"], "|", m["title"], "|", m.get("open_time"))

In [ ]:
def kalshi_get(path, params=None):
    resp = requests.get(
        KALSHI_BASE + path,
        params=params,
        timeout=30
    )
    resp.raise_for_status()
    return resp.json()

In [ ]:
# Verify all tickers
print("=== SERIES TICKERS ===")
for series_key, config in PLAYOFF_SERIES.items():
    for team, ticker in config["teams"].items():
        resp = requests.get(f"{KALSHI_BASE}/markets/{ticker}")
        if resp.status_code == 200:
            market = resp.json().get("market", {})
            print(f"✅ {ticker} | {market.get('title')} | yes_bid={market.get('yes_bid_dollars')}")
        else:
            print(f"❌ {ticker} | {resp.status_code}")

print("\n=== GAME TICKERS ===")
for series_key, config in PLAYOFF_SERIES.items():
    for date, game_info in config["game_tickers"].items():
        ticker, start = game_info[0], game_info[1]
        resp = requests.get(f"{KALSHI_BASE}/markets/{ticker}")
        if resp.status_code == 200:
            market = resp.json().get("market", {})
            print(f"✅ {ticker} | {market.get('title')} | open={market.get('open_time')} | close={market.get('close_time')}")
        else:
            print(f"❌ {ticker} | {resp.status_code}")

In [ ]:
def get_kalshi_candles(ticker, start_dt, end_dt, period_interval=1):
    series = ticker.split("-")[0]
    path   = f"/series/{series}/markets/{ticker}/candlesticks"
    
    all_candles = []
    current = start_dt
    
    while current < end_dt:
        window_end = min(current + timedelta(hours=72), end_dt)
        params = {
            "start_ts":        int(current.timestamp()),
            "end_ts":          int(window_end.timestamp()),
            "period_interval": period_interval,
        }
        try:
            resp = requests.get(f"{KALSHI_BASE}{path}", params=params, timeout=30)
            resp.raise_for_status()
            candles = resp.json().get("candlesticks", [])
            all_candles.extend(candles)
        except Exception as e:
            print(f"  WARNING: {e}")
        
        current = window_end
        time.sleep(0.3)
    
    if not all_candles:
        return pd.Series(dtype=float)
    
    df = pd.DataFrame(all_candles)
    df["time"] = pd.to_datetime(df["end_period_ts"], unit="s", utc=True)
    df = df.set_index("time").sort_index()
    
    # extract close_dollars from nested yes_ask dict
    # yes_ask is the best ask price = what you'd pay to buy YES
    def extract_close(x):
        if isinstance(x, dict):
            val = x.get("close_dollars")
            if val is not None:
                return float(val)
        return float("nan")
    
    prices = df["yes_ask"].apply(extract_close)
    prices = prices.dropna()
    
    if prices.empty:
        print(f"  WARNING: no valid prices extracted")
        return pd.Series(dtype=float)
    
    return prices.resample("1min").last().ffill()

In [ ]:
def get_kalshi_game_window(game_ticker):
    """Get start/end times for a game from Kalshi API."""
    path = f"/markets/{game_ticker}"
    try:
        resp   = requests.get(f"{KALSHI_BASE}{path}", timeout=30)
        resp.raise_for_status()
        market = resp.json().get("market", {})
        
        start = pd.to_datetime(market["open_time"])
        if start.tzinfo is None:
            start = start.tz_localize("UTC")
        
        # use expected_expiration_time as game end if available
        end_field = market.get("expected_expiration_time") or market.get("close_time")
        end = pd.to_datetime(end_field)
        if end.tzinfo is None:
            end = end.tz_localize("UTC")
        end = end + timedelta(minutes=10)
        
        print(f"  Window: {start} -> {end}")
        return start, end
    
    except Exception as e:
        print(f"  WARNING getting window for {game_ticker}: {e}")
        return None, None

In [ ]:
def build_series_df(series_key):
    config = PLAYOFF_SERIES[series_key]
    
    # unpack first game tuple — (ticker, start)
    first_ticker, series_start = list(config["game_tickers"].values())[0]
    series_end = datetime.now(timezone.utc)

    print(f"\nBuilding series: {series_key} | {series_start.date()} -> {series_end.date()}")

    cols = {}
    for short_name, ticker in config["teams"].items():
        print(f"  Fetching {short_name} ({ticker})...")
        prices = get_kalshi_candles(ticker, series_start, series_end)
        if not prices.empty:
            cols[short_name] = prices

    if not cols:
        print("  WARNING: no data fetched")
        return pd.DataFrame()

    return pd.DataFrame(cols).ffill()

In [ ]:
def build_games_df(series_key):
    config = PLAYOFF_SERIES[series_key]
    print(f"\nBuilding games: {series_key}")
    
    game_dfs     = {}
    game_windows = {}
    
    for date, (game_ticker, start) in config["game_tickers"].items():
        print(f"\n  Game: {date} ({game_ticker})")
        
        try:
            # get end time from API close_time
            resp   = requests.get(f"{KALSHI_BASE}/markets/{game_ticker}", timeout=30)
            market = resp.json().get("market", {})
            end    = pd.to_datetime(market["close_time"])
            if end.tzinfo is None:
                end = end.tz_localize("UTC")
            end = end + timedelta(minutes=10)
            
            print(f"  Window: {start} -> {end}")
            
            prices_yes = get_kalshi_candles(game_ticker, start, end)
            
            if not prices_yes.empty:
                home_code    = game_ticker.split("-")[-1]
                code_to_name = {"NYK": "Knicks", "CLE": "Cavaliers",
                                "OKC": "Thunder", "SAS": "Spurs"}
                home_team = code_to_name.get(home_code, home_code)
                away_team = [t for t in config["teams"].keys() if t != home_team][0]
                
                cols = {
                    home_team: prices_yes,
                    away_team: 1 - prices_yes,
                }
                
                game_dfs[date]     = pd.DataFrame(cols).ffill()
                game_windows[date] = {"start": start, "end": end}
        
        except Exception as e:
            print(f"  Skipping {game_ticker}: {e}")
    
    if not game_dfs:
        return pd.DataFrame(), {}
    
    combined = pd.concat(game_dfs, axis=1)
    combined.columns.names = ["game", "team"]
    return combined, game_windows

In [ ]:
# Series data
all_series_dfs = {}
for series_key in PLAYOFF_SERIES:
    all_series_dfs[series_key] = build_series_df(series_key)

# Combine into MultiIndex series panel
series_panel = pd.concat(all_series_dfs, axis=1)
series_panel.columns.names = ["series", "team"]
print("\nSeries panel:")
print(series_panel.shape)
print(series_panel.head())

# Game data
all_games_dfs    = {}
all_game_windows = {}
for series_key in PLAYOFF_SERIES:
    all_games_dfs[series_key], all_game_windows[series_key] = build_games_df(series_key)

print("\nGame panels:")
for k, v in all_games_dfs.items():
    print(f"  {k}: {v.shape}")
    print(f"  Games: {v.columns.get_level_values('game').unique().tolist()}")
    print(f"  Teams: {v.columns.get_level_values('team').unique().tolist()}")

In [ ]:
#So we don't have overlapping game windows
def add_game_buffers(combined_df, games_df, buffer_mins=5):
    result = combined_df.copy()
    for game_key in games_df.columns.get_level_values("game").unique():
        game_data = games_df[game_key]
        active = game_data[game_data.sum(axis=1) > 0]
        if active.empty:
            continue
        game_start = active.index.min()
        game_end   = active.index.max()
        buffer_before = pd.DataFrame(
            float("nan"),
            index=pd.date_range(end=game_start - timedelta(minutes=1), periods=buffer_mins, freq="1min", tz="UTC"),
            columns=result.columns
        )
        buffer_after = pd.DataFrame(
            float("nan"),
            index=pd.date_range(start=game_end + timedelta(minutes=1), periods=buffer_mins, freq="1min", tz="UTC"),
            columns=result.columns
        )
        result = pd.concat([result, buffer_before, buffer_after]).sort_index()
        result = result[~result.index.duplicated(keep="first")]
    return result


def get_game_windows(games_df):
    game_windows = []
    for game_key in games_df.columns.get_level_values("game").unique():
        game_data = games_df[game_key]
        active = game_data[game_data.sum(axis=1) > 0]
        if active.empty:
            continue
        game_windows.append((active.index.min(), active.index.max()))
    return sorted(game_windows, key=lambda x: x[0])

#So we only extract signa/trade when the game is occurring
def slice_series_to_games(series_df, games_df, buffer_mins=5):
    game_windows = get_game_windows(games_df)
    pieces = []
    for i, (start, end) in enumerate(game_windows):
        game_slice = series_df[start:end]
        if game_slice.empty:
            continue
        pieces.append(game_slice)
        if i < len(game_windows) - 1:
            buffer_idx = pd.date_range(
                start=end + timedelta(minutes=1),
                periods=buffer_mins, freq="1min", tz="UTC"
            )
            buffer = pd.DataFrame(float("nan"), index=buffer_idx, columns=series_df.columns)
            pieces.append(buffer)
    if not pieces:
        return pd.DataFrame()
    result = pd.concat(pieces).sort_index()
    result = result[~result.index.duplicated(keep="first")]
    return result

In [ ]:
# Stack all games into 2 columns
ecf_games_combined = pd.DataFrame({
    "Knicks":    all_games_dfs["ECF_NYK_CLE"].xs("Knicks",    axis=1, level="team").sum(axis=1),
    "Cavaliers": all_games_dfs["ECF_NYK_CLE"].xs("Cavaliers", axis=1, level="team").sum(axis=1),
})
wcf_games_combined = pd.DataFrame({
    "Thunder": all_games_dfs["WCF_OKC_SAS"].xs("Thunder", axis=1, level="team").sum(axis=1),
    "Spurs":   all_games_dfs["WCF_OKC_SAS"].xs("Spurs",   axis=1, level="team").sum(axis=1),
})

ecf_games_combined = add_game_buffers(ecf_games_combined, all_games_dfs["ECF_NYK_CLE"])
wcf_games_combined = add_game_buffers(wcf_games_combined, all_games_dfs["WCF_OKC_SAS"])

# Slice series to game windows
ecf_series = slice_series_to_games(all_series_dfs["ECF_NYK_CLE"], all_games_dfs["ECF_NYK_CLE"])
wcf_series = slice_series_to_games(all_series_dfs["WCF_OKC_SAS"], all_games_dfs["WCF_OKC_SAS"])

Part 2: In depth look at the strategy

We get the returns during each one minute interval for each game

In [ ]:
# Compute rets
knicks_cavs_game_rets  = ecf_games_combined / ecf_games_combined.shift() - 1
ecf_series_rets        = ecf_series / ecf_series.shift() - 1
thunder_spurs_game_rets = wcf_games_combined / wcf_games_combined.shift() - 1
wcf_series_rets        = wcf_series / wcf_series.shift() - 1

Based on the rolling game returns over 1-10 minutes and differing thresholds for when we trade for each individual team, we trade a momentum signal on the overall series to determine how many periods we should base the momentum signal off of and what our threshold should be. 

Mathematically,

$$s_t = \frac{1}{k} \sum_{i=0}^{k-1} r_{t-i}^{\text{game}}$$

$$p_t = \text{sign}(s_{t-1}) \cdot \mathbf{1}\left[|s_{t-1}| > \theta\right]$$

$$\text{PnL}_t = r_t^{\text{series}} \cdot p_t$$

where $k \in \{1, \ldots, 10\}$ is the lookback window in minutes.

In [ ]:
TEAM_DATA = {
    "Knicks":    {"game_rets": knicks_cavs_game_rets["Knicks"],    "series_rets": ecf_series_rets["Knicks"]},
    "Cavaliers": {"game_rets": knicks_cavs_game_rets["Cavaliers"], "series_rets": ecf_series_rets["Cavaliers"]},
    "Thunder":   {"game_rets": thunder_spurs_game_rets["Thunder"], "series_rets": wcf_series_rets["Thunder"]},
    "Spurs":     {"game_rets": thunder_spurs_game_rets["Spurs"],   "series_rets": wcf_series_rets["Spurs"]},
}

rates      = list(range(1, 11))
thresholds = [round(x, 3) for x in np.arange(0.005, 0.065, 0.005)]
#Assume zero fees at the moment
fee_multiplier = 0

all_strats = {}

for team, data in TEAM_DATA.items():
    game_rets   = data["game_rets"]
    series_rets = data["series_rets"]
    cols = {}

    for rate in rates:
        ret = game_rets.rolling(int(rate), min_periods=1).mean()
        for threshold in thresholds:
            port = pd.Series(0.0, index=ret.index)
            port[ret >  threshold] =  1.0
            port[ret < -threshold] = -1.0
            port = port.shift() * series_rets
            cols[(rate, threshold)] = port

    df = pd.DataFrame(cols)
    df.columns = pd.MultiIndex.from_tuples(df.columns, names=["rate", "threshold"])
    all_strats[team] = df

# ── Sharpe surface ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

sharpe_grids = {}

for i, (team, df) in enumerate(all_strats.items()):
    grid = np.zeros((len(rates), len(thresholds)))
    for ri, rate in enumerate(rates):
        for ti, thr in enumerate(thresholds):
            pnl = df[(rate, thr)].dropna()
            if len(pnl) > 50:
                grid[ri, ti] = pnl.mean() / pnl.std() * np.sqrt(150)

    sharpe_grids[team] = pd.DataFrame(grid, index=rates, columns=thresholds)

    im = axes[i].imshow(
        grid, aspect="auto", origin="lower",
        extent=[thresholds[0], thresholds[-1], rates[0], rates[-1]],
        cmap="RdYlGn", vmin=-1, vmax=3
    )
    plt.colorbar(im, ax=axes[i], label="Sharpe")
    axes[i].set_title(f"{team} — Sharpe over (rate, threshold)")
    axes[i].set_xlabel("threshold")
    axes[i].set_ylabel("lookback (bars)")

    best_idx = np.unravel_index(np.argmax(grid), grid.shape)
    best_rate, best_thr = rates[best_idx[0]], thresholds[best_idx[1]]
    axes[i].scatter(best_thr, best_rate, color="blue", s=80, zorder=5,
                    label=f"best: rate={best_rate}, thr={best_thr}")
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\nBest (rate, threshold) per team:")
for team, grid_df in sharpe_grids.items():
    best = grid_df.stack().idxmax()
    print(f"  {team:12s} rate={best[0]:2d}  threshold={best[1]:.3f}  Sharpe={grid_df.stack().max():.3f}")

The lower left corner returns high sharpes on average so we decide to set our threshold at 0.01. Just to be sure, we test the momentum signal on rolling game returns from 1-10 to see the exact Sharpes

In [ ]:
TEAM_DATA = {
    "Knicks":    {"game_rets": knicks_cavs_game_rets["Knicks"],    "series_rets": ecf_series_rets["Knicks"]},
    "Cavaliers": {"game_rets": knicks_cavs_game_rets["Cavaliers"], "series_rets": ecf_series_rets["Cavaliers"]},
    "Thunder":   {"game_rets": thunder_spurs_game_rets["Thunder"], "series_rets": wcf_series_rets["Thunder"]},
    "Spurs":     {"game_rets": thunder_spurs_game_rets["Spurs"],   "series_rets": wcf_series_rets["Spurs"]},
}

rates = [1, 2, 3, 4, 5, 6,7,8,9,10]

all_strats = {}
#Threshold
threshold = 0.01
#hold off on computing fees for the moment
fee_multiplier = 0

for team, data in TEAM_DATA.items():
    game_rets   = data["game_rets"]
    series_rets = data["series_rets"]
    team_strats = {}
    for rate in rates:
        ret  = game_rets.rolling(int(rate), min_periods=1).mean()
        #check how many above/below threshold
        above = (ret.abs() > 0.01).sum()
        zeros = (ret == 0).sum()
        print(f"Rate {rate}: above threshold={above}, zeros={zeros}")
        
        port = pd.Series(0, index=ret.index)  # default 0
        port[ret >  threshold] =  1
        port[ret < -threshold] = -1
        
        port = port.shift() * series_rets
        team_strats[str(rate)] = port
    all_strats[team] = pd.DataFrame(team_strats)

game_window = ("2026-05-27 00:00", "2026-05-27 04:00")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, (team, strats) in enumerate(all_strats.items()):
    strats[game_window[0]:game_window[1]].cumsum().plot(
        ax=axes[i],
        title=f"{team} - Momentum Strategy (rates 1-10)"
    )
    axes[i].set_xlabel("time")
    axes[i].set_ylabel("cumulative return")

plt.tight_layout()
plt.show()

In [ ]:
#check number of zeros --> How many trades we make for a team
for i in range(10):
    s = all_strats['Thunder'][str(i+1)].dropna()
    
    raw_signal = s.copy()
    raw_signal[s != 0] = 1  
    
    # every transition is a fee: 0→1, 1→0, +1→-1 etc.
    fee_events = (raw_signal != raw_signal.shift()).sum()
    print(f"Rate {i+1}: {fee_events} fee events")


Compute the Sharpe Ratio for each team to find an optimal rolling mean

In [ ]:
sharpes = {}
for key in all_strats.keys():
    rets = all_strats[key]
    rets = rets.dropna()
    sr = rets.mean() / rets.std() * np.sqrt(150)
    sharpes[key] = sr

sharpes = pd.DataFrame(sharpes)
print(sharpes)
print("\nMean across rates:")
print(sharpes.mean(axis=1))

We find that using the mean over 3 minutes gives us the best Sharpe Ratio.

The threshold of 0.01 mean that we don't trade based on any 3 minute average movements that have an absolute value less than 0.01. To add more intuition behind why this threshold produces high sharpes, we present the following result where we show that a threshold of 0.01 is roughly equivalent to keeping the top 80% of signals when using a rolling mean of 3 minute periods for our signal.

In [ ]:
all_rets = pd.concat([knicks_cavs_game_rets, thunder_spurs_game_rets], axis =0)
all_rets = all_rets[all_rets < 20]
all_rets = all_rets[all_rets != 0]
all_rets  = all_rets.stack()
all_rets = all_rets.rolling(3).mean()

#look at the distribution of the absolute value of rets
all_rets.abs().plot(kind='hist', bins=100, title='Distribution of Game Returns')

#take the top 80% of values
print("We keep roughly the top 80% of signals")
(all_rets.abs() >= 0.01).mean()

While other thresholds were tested and returned higher Sharpe Ratios, 0.01 was still chosen for reasons explained later in the notebook 

Now we test a combined strategy where we trade both teams simultaneously with even weighting during games. We use a rolling mean of 3. We do this to potentially increase the Sharpe Ratio

In [ ]:
combined_games = {}
combined_games['Knicks'] = knicks_cavs_game_rets['Knicks']
combined_games['Thunder'] = thunder_spurs_game_rets['Thunder']
combined_games['Cavaliers'] = knicks_cavs_game_rets['Cavaliers']
combined_games['Spurs'] = thunder_spurs_game_rets['Spurs']
combined_games = pd.DataFrame(combined_games)

combined_series = {}
combined_series['Knicks'] = ecf_series_rets['Knicks']
combined_series['Thunder'] = wcf_series_rets['Thunder']
combined_series['Cavaliers'] = ecf_series_rets['Cavaliers']
combined_series['Spurs'] = wcf_series_rets['Spurs']
combined_series = pd.DataFrame(combined_series)

threshold = 0.01
ret = combined_games.rolling(int(3), min_periods=1).mean()
#print(ret)
port = pd.DataFrame(0, index=ret.index,columns=ret.columns)  # default 0
port[ret >  threshold] =  1
port[ret < -threshold] = -1

denom = port.abs().sum(axis=1)
print(denom.value_counts(normalize=True))
 # exactly +/-0.5 each
denom = port.abs().sum(axis=1).replace(0, np.nan)
port = port.divide(denom, axis=0).fillna(0)

port_saved = port

port = port.shift() * combined_series


total_rets = port.sum(axis=1, min_count=1)


print(total_rets["2026-05-25 00:00": "2026-05-25 04:00"].cumsum().plot())

In [ ]:
#Number of trades
position_changes = (port_saved != port_saved.shift())
total_trades = position_changes.sum().sum()
print(f"Total trades: {total_trades}")
print(f"Total trades per game: {total_trades/9}")

In [ ]:
sr = total_rets.dropna().mean()/total_rets.dropna().std()*np.sqrt(150)
sr


The in-sample t-stat is 7.15 (n=~1500)

In [ ]:
#Calculate t-stat for average returns being 0 
print("t-stat")
total_rets.dropna().mean()/total_rets.dropna().std()*np.sqrt(len(total_rets))

Total Returns

In [ ]:
cum = total_rets.dropna().cumsum()
print(f"Cumulative Returns: {cum[-1]}")


Max Drawdown

In [ ]:
cumulative  = total_rets.cumsum()
running_max = cumulative.cummax()
drawdown    = cumulative - running_max
print(f"Max Drawdown: {drawdown.min()}")


In [ ]:
# Hit rate — fraction of non-zero minutes where we made money
nonzero = total_rets[total_rets != 0].dropna()
hit_rate = (nonzero > 0).mean()
print(f"Overall hit rate (non-zero minutes): {hit_rate:.2%}")

In [ ]:
# Split total_rets into per-game chunks at NaN boundaries
game_chunks = []
current_chunk = []

for val in total_rets.items():
    idx, v = val
    if pd.isna(v):
        if len(current_chunk) > 0:
            game_chunks.append(pd.Series(dict(current_chunk)))
            current_chunk = []
    else:
        current_chunk.append((idx, v))

# Don't forget the last chunk
if len(current_chunk) > 0:
    game_chunks.append(pd.Series(dict(current_chunk)))

print(f"Number of games found: {len(game_chunks)}")
for i, chunk in enumerate(game_chunks):
    print(f"  Game {i+1}: {chunk.index[0]} -> {chunk.index[-1]}  (n={len(chunk)})")

We also report per game statistics for returns, sharpe, hit-rate, and drawdowns.

In [ ]:
print("=" * 72)
print(f"{'Game':<8} {'Start':<22} {'Cum Ret':>10} {'Sharpe':>8} {'Hit Rate':>10} {'Max DD':>10}")
print("-" * 72)
for i, chunk in enumerate(game_chunks):
    nonzero = chunk[chunk != 0]
    hr      = (nonzero > 0).mean() if len(nonzero) > 0 else float("nan")
    cum_ret = chunk.sum()
    sr      = chunk.mean() / chunk.std() * np.sqrt(150) if chunk.std() > 0 else float("nan")
    cum     = chunk.cumsum()
    dd      = (cum - cum.cummax()).min()
    start   = str(chunk.index[0])[:16]
    print(f"{'G'+str(i+1):<8} {start:<22} {cum_ret:>10.4f} {sr:>8.3f} {hr:>10.2%} {dd:>10.4f}")
print("=" * 72)

# Summary row
all_nonzero = total_rets[total_rets != 0].dropna()
overall_hr  = (all_nonzero > 0).mean()
overall_ret = total_rets.dropna().sum()
overall_sr  = total_rets.dropna().mean() / total_rets.dropna().std() * np.sqrt(150)
cum_all     = total_rets.dropna().cumsum()
overall_dd  = (cum_all - cum_all.cummax()).min()
print(f"{'TOTAL':<8} {'':<22} {overall_ret:>10.4f} {overall_sr:>8.3f} {overall_hr:>10.2%} {overall_dd:>10.4f}")
print("=" * 72)

Test on unseen Game 6 of WCF and Games 1-4 of the NBA Finals.

In [ ]:
PLAYOFF_SERIES_OOS = {
    "WCF_OKC_SAS": {
        "teams": {
            "Thunder": "KXNBAWEST-26-OKC",
            "Spurs":   "KXNBAWEST-26-SAS",
        },
        "game_tickers": {
            "2026-05-28": ("KXNBAGAME-26MAY28OKCSAS-SAS", datetime(2026, 5, 29, 0, 30, tzinfo=timezone.utc)),
        }
    },
    "FINALS_NYK_SAS": {
        "teams": {
            "Knicks": "KXNBA-26-NYK",
            "Spurs":  "KXNBA-26-SAS",
        },
        "game_tickers": {
            # Game 1 — SAS home (NYK @ SAS). Tip ~8:30 PM ET → 00:30 UTC next day	
            "2026-06-03": ("KXNBAGAME-26JUN03NYKSAS-SAS", datetime(2026, 6, 4, 0, 30, tzinfo=timezone.utc)),
            # Game 2 — SAS home
            "2026-06-05": ("KXNBAGAME-26JUN05NYKSAS-NYK", datetime(2026, 6, 6, 0, 30, tzinfo=timezone.utc)),
            # Game 3 — NYK home 
            "2026-06-8": ("KXNBAGAME-26JUN08SASNYK-NYK", datetime(2026, 6, 9, 0, 30, tzinfo=timezone.utc)),
            #Game 4  - NYK home 
            "2026-06-10": ("KXNBAGAME-26JUN10SASNYK-NYK", datetime(2026, 6, 11, 0, 30, tzinfo=timezone.utc)),

            "2026-06-13": ("KXNBAGAME-26JUN13NYKSAS-SAS", datetime(2026, 6, 14, 0, 30, tzinfo=timezone.utc)),
        }
    }
}

In [ ]:
# ── Fetch data ─────────────────────────────────────────────────────────────────
# Temporarily swap global so existing build functions work unchanged
_backup = PLAYOFF_SERIES
PLAYOFF_SERIES = PLAYOFF_SERIES_OOS

all_series_dfs_oos   = {}
all_games_dfs_oos    = {}
all_game_windows_oos = {}

for series_key in PLAYOFF_SERIES_OOS:
    all_series_dfs_oos[series_key] = build_series_df(series_key)
    all_games_dfs_oos[series_key], all_game_windows_oos[series_key] = build_games_df(series_key)

PLAYOFF_SERIES = _backup  # restore immediately after fetching

# ── Stack into per-series price frames ────────────────────────────────────────
wcf_games_oos = pd.DataFrame({
    "Thunder": all_games_dfs_oos["WCF_OKC_SAS"].xs("Thunder", axis=1, level="team").sum(axis=1),
    "Spurs":   all_games_dfs_oos["WCF_OKC_SAS"].xs("Spurs",   axis=1, level="team").sum(axis=1),
})
finals_games_oos = pd.DataFrame({
    "Knicks": all_games_dfs_oos["FINALS_NYK_SAS"].xs("Knicks", axis=1, level="team").sum(axis=1),
    "Spurs":  all_games_dfs_oos["FINALS_NYK_SAS"].xs("Spurs",  axis=1, level="team").sum(axis=1),
})

wcf_games_oos    = add_game_buffers(wcf_games_oos,    all_games_dfs_oos["WCF_OKC_SAS"])
finals_games_oos = add_game_buffers(finals_games_oos, all_games_dfs_oos["FINALS_NYK_SAS"])

wcf_series_oos    = slice_series_to_games(all_series_dfs_oos["WCF_OKC_SAS"],    all_games_dfs_oos["WCF_OKC_SAS"])
finals_series_oos = slice_series_to_games(all_series_dfs_oos["FINALS_NYK_SAS"], all_games_dfs_oos["FINALS_NYK_SAS"])

# ── Compute returns ────────────────────────────────────────────────────────────
wcf_game_rets_oos   = wcf_games_oos / wcf_games_oos.shift() - 1
wcf_series_rets_oos = wcf_series_oos / wcf_series_oos.shift() - 1
fin_game_rets_oos   = finals_games_oos / finals_games_oos.shift() - 1
fin_series_rets_oos = finals_series_oos / finals_series_oos.shift() - 1

# Disambiguate Spurs — WCF and Finals are different contracts
wcf_game_rets_oos   = wcf_game_rets_oos.rename(columns={"Spurs": "Spurs_WCF"})
wcf_series_rets_oos = wcf_series_rets_oos.rename(columns={"Spurs": "Spurs_WCF"})

In [ ]:
combined_games_oos = {}
combined_games_oos['Thunder']   = wcf_game_rets_oos['Thunder']
combined_games_oos['Spurs_WCF'] = wcf_game_rets_oos['Spurs_WCF']
combined_games_oos['Knicks']    = fin_game_rets_oos['Knicks']
combined_games_oos['Spurs']     = fin_game_rets_oos['Spurs']
combined_games_oos = pd.DataFrame(combined_games_oos)

combined_series_oos = {}
combined_series_oos['Thunder']   = wcf_series_rets_oos['Thunder']
combined_series_oos['Spurs_WCF'] = wcf_series_rets_oos['Spurs_WCF']
combined_series_oos['Knicks']    = fin_series_rets_oos['Knicks']
combined_series_oos['Spurs']     = fin_series_rets_oos['Spurs']
combined_series_oos = pd.DataFrame(combined_series_oos)

threshold = 0.01
ret = combined_games_oos.rolling(int(3), min_periods=1).mean()
port = pd.DataFrame(0, index=ret.index, columns=ret.columns)
port[ret >  threshold] =  1
port[ret < -threshold] = -1

denom = port.abs().sum(axis=1)
denom = port.abs().sum(axis=1).replace(0, np.nan)
port = port.divide(denom, axis=0).fillna(0)

port_saved = port.copy()

port = port.shift() * combined_series_oos

total_rets_oos = port.sum(axis=1, min_count=1)

print(total_rets_oos.isna().sum())

# NBA Finals Game 1
print(total_rets_oos["2026-06-14 00:00":"2026-06-14 04:00"].cumsum().plot(title="NBA Finals Game 1"))


In [ ]:
#Cumsum returns
print(total_rets_oos.dropna().cumsum().plot())

Sharpe Ratio and T-Stat

In [ ]:
n_oos  = len(total_rets_oos)
sr_oos = total_rets_oos.mean() / total_rets_oos.std() * np.sqrt(150)
t_oos  = total_rets_oos.mean() / total_rets_oos.std() * np.sqrt(n_oos)
p_oos  = 2 * (1 - stats.t.cdf(abs(t_oos), df=n_oos - 1))

print(f"OOS Sharpe : {sr_oos:.3f}")
print(f"OOS t-stat : {t_oos:.2f}  (n={n_oos},  p={p_oos:.4f})")

As we can see, the Sharpe Ratio maintains much of its potency on the held out games. While there is not enough data to declare this strategy always has a trong Sharpe Ratio, the results are promising enough to attempt live trading. 

Total Cumulative Returns

In [ ]:
cum = total_rets_oos.dropna().cumsum()
print(f"Cumulative Returns: {cum[-1]}")

Max Drawdown

In [ ]:
cumulative  = total_rets_oos.cumsum()
running_max = cumulative.cummax()
drawdown    = cumulative - running_max
print(f"Max Drawdown: {drawdown.min()}")


In [ ]:
# Hit rate — fraction of non-zero minutes where we made money
nonzero = total_rets_oos[total_rets_oos != 0].dropna()
hit_rate = (nonzero > 0).mean()
print(f"Overall hit rate (non-zero minutes): {hit_rate:.2%}")

In [ ]:
# Split total_rets into per-game chunks at NaN boundaries
game_chunks = []
current_chunk = []

for val in total_rets_oos.items():
    idx, v = val
    if pd.isna(v):
        if len(current_chunk) > 0:
            game_chunks.append(pd.Series(dict(current_chunk)))
            current_chunk = []
    else:
        current_chunk.append((idx, v))

# Don't forget the last chunk
if len(current_chunk) > 0:
    game_chunks.append(pd.Series(dict(current_chunk)))

print(f"Number of games found: {len(game_chunks)}")
for i, chunk in enumerate(game_chunks):
    print(f"  Game {i+1}: {chunk.index[0]} -> {chunk.index[-1]}  (n={len(chunk)})")

In [ ]:
print("=" * 72)
print(f"{'Game':<8} {'Start':<22} {'Cum Ret':>10} {'Sharpe':>8} {'Hit Rate':>10} {'Max DD':>10}")
print("-" * 72)
for i, chunk in enumerate(game_chunks):
    nonzero = chunk[chunk != 0]
    hr      = (nonzero > 0).mean() if len(nonzero) > 0 else float("nan")
    cum_ret = chunk.sum()
    sr      = chunk.mean() / chunk.std() * np.sqrt(150) if chunk.std() > 0 else float("nan")
    cum     = chunk.cumsum()
    dd      = (cum - cum.cummax()).min()
    start   = str(chunk.index[0])[:16]
    print(f"{'G'+str(i+1):<8} {start:<22} {cum_ret:>10.4f} {sr:>8.3f} {hr:>10.2%} {dd:>10.4f}")
print("=" * 72)

# Summary row
all_nonzero = total_rets_oos[total_rets_oos != 0].dropna()
overall_hr  = (all_nonzero > 0).mean()
overall_ret = total_rets_oos.dropna().sum()
overall_sr  = total_rets_oos.dropna().mean() / total_rets_oos.dropna().std() * np.sqrt(150)
cum_all     = total_rets_oos.dropna().cumsum()
overall_dd  = (cum_all - cum_all.cummax()).min()
print(f"{'TOTAL':<8} {'':<22} {overall_ret:>10.4f} {overall_sr:>8.3f} {overall_hr:>10.2%} {overall_dd:>10.4f}")
print("=" * 72)

In [ ]:
# Series data
all_series_dfs = {}
for series_key in PLAYOFF_SERIES:
    all_series_dfs[series_key] = build_series_df(series_key)

# Combine into MultiIndex series panel
series_panel = pd.concat(all_series_dfs, axis=1)
series_panel.columns.names = ["series", "team"]
print("\nSeries panel:")
print(series_panel.shape)
print(series_panel.head())

# Game data
all_games_dfs    = {}
all_game_windows = {}
for series_key in PLAYOFF_SERIES:
    all_games_dfs[series_key], all_game_windows[series_key] = build_games_df(series_key)

print("\nGame panels:")
for k, v in all_games_dfs.items():
    print(f"  {k}: {v.shape}")
    print(f"  Games: {v.columns.get_level_values('game').unique().tolist()}")
    print(f"  Teams: {v.columns.get_level_values('team').unique().tolist()}")

In [ ]:
# Stack all games into 2 columns
wcf_games_combined_test = pd.DataFrame({
    "Thunder": all_games_dfs["WCF_OKC_SAS"].xs("Thunder", axis=1, level="team").sum(axis=1),
    "Spurs":   all_games_dfs["WCF_OKC_SAS"].xs("Spurs",   axis=1, level="team").sum(axis=1),
})

wcf_games_combined_test = add_game_buffers(wcf_games_combined_test, all_games_dfs["WCF_OKC_SAS"])

# Slice series to game windows
wcf_series_test = slice_series_to_games(all_series_dfs["WCF_OKC_SAS"], all_games_dfs["WCF_OKC_SAS"])

In [ ]:
# Compute rets
thunder_spurs_game_rets_test = wcf_games_combined_test / wcf_games_combined_test.shift() - 1
wcf_series_rets_test        = wcf_series_test / wcf_series_test.shift() - 1

In [ ]:
combined_games_test = {}
combined_games_test['Thunder'] = thunder_spurs_game_rets_test['Thunder']
combined_games_test['Spurs'] = thunder_spurs_game_rets_test['Spurs']
combined_games_test = pd.DataFrame(combined_games_test)

combined_series_test = {}
combined_series_test['Thunder'] = wcf_series_rets_test['Thunder']
combined_series_test['Spurs'] = wcf_series_rets_test['Spurs']
combined_series_test = pd.DataFrame(combined_series_test)

threshold = 0.01
ret = combined_games_test.rolling(int(3), min_periods=1).mean()
#print(ret)
port = pd.DataFrame(0, index=ret.index,columns=ret.columns)  # default 0
port[ret >  threshold] =  1
port[ret < -threshold] = -1

denom = port.abs().sum(axis=1)
print(denom.value_counts(normalize=True))
 # exactly +/-0.5 each
denom = port.abs().sum(axis=1).replace(0, np.nan)
port = port.divide(denom, axis=0).fillna(0)

port_saved = port

port = port.shift() * combined_series_test

total_rets = port.sum(axis=1, min_count=1)

print(total_rets.shape)
print(total_rets.isna().sum())
print(total_rets["2026-05-29 00:00": "2026-05-29 04:00"].cumsum().plot())

In [ ]:
sr = total_rets.mean()/total_rets.std()*np.sqrt(150)
sr

The out-of-sample t-stat is 1.98 (n=~150)

In [ ]:
#Calculate t-stat for average returns being 0 
total_rets = total_rets.dropna()
print("t-stat")
total_rets.mean()/total_rets.std()*np.sqrt(len(total_rets))

As we can see, the strong sharpe ratio is maintained on the test set giving us confidence to begin to apply it to a live setting.

Part 3: Accounting for Fees

We estimate fees on market orders on Kalshi with the formula fee=0.07×C×(1−C) per contract where C is the price. We also smooth out the signal by only trading when the signal flipped to 0 or the other direction from long to short. This should reduce the number of times we trade and as a result the fees.

In [ ]:
#Get prices
combined_prices = pd.DataFrame({
    'Knicks':    ecf_games_combined['Knicks'],
    'Cavaliers': ecf_games_combined['Cavaliers'],
    'Thunder':   wcf_games_combined['Thunder'],
    'Spurs':     wcf_games_combined['Spurs'],
})

threshold = 0.01
fee_multiplier = 0.07

ret = combined_games.rolling(int(3), min_periods=1).mean()

port = pd.DataFrame(0, index=ret.index, columns=ret.columns)
port[ret >  threshold] =  1
port[ret < -threshold] = -1

# fee on clean signal changes only
fee_cost = (port - port.shift()).abs() * fee_multiplier * combined_prices * (1 - combined_prices)

port_saved = port

# execute strategy — shift port only, not fee_cost
port = port.shift() * combined_series
total_rets = port - fee_cost
total_rets = total_rets.sum(axis=1, min_count=1).dropna()

print(f"Total fee drag: {fee_cost.sum().sum():.4f}")
print(f"Net Sharpe: {total_rets.mean() / total_rets.std() * np.sqrt(150):.3f}")
print(total_rets.cumsum().plot())

In [ ]:
#Number of trades
position_changes = (port_saved != port_saved.shift())
total_trades = position_changes.sum().sum()
print(f"Total trades: {total_trades}")
print(f"Total trades per game: {np.floor(total_trades/9)} is less than original of 120")

In [ ]:
sr = total_rets.mean()/total_rets.std()*np.sqrt(150)
sr

Clearly the excessive fees on Kalshi, eat away into the Sharpe Ratio. This is a known fact about Kalshi and I suspect it is because they want to encourage people to be market makers and thus increase their liquidity as they try to grow

This begs the question. How do we preserve our Sharpe in the face of excessive fees? First, the threshold was adjusted to a larger value such as 0.04 to encourage fewer trades. Even though this actually increased the Sharpe before fees and reduced the number of trades, the fees were still disastrous to the strategy. We leave it out for the sake of brevity

Part 4: Utilizing Limit Orders to Reduce Fees

Limit orders have significantly lower fees on Kalshi so we trade them as follows. 

If our signal says to go long, we place a limit order at ask_price - 1 and if it fills we hold it until the signal flips. If our signal says to go short, we place a limit order at ask_price + 1 and hold it until the signal flips, at which point we buy back our shares and capture the difference as our profit

In [ ]:
#Sample code from a trading library (WSQ) adapted to Kalshi. Not meant to run here. 
def create_order(self, ticker, qty, side, limit_price=None, custom_id=None, route=False):
    if limit_price is None:
        px = self.get_current_price(ticker)
        if np.isnan(px) or px <= 0:
            return
        limit_price = float(
            np.clip(
                px - self._limit_offset if side == "buy" else px + self._limit_offset,
                0.01,
                0.99,
            )
        )
    self.broker.place_limit_order(ticker, side, float(qty), float(limit_price), custom_id)

def trade_to_target(self, target: Dict[str, float], route: bool = False):
    """Rebalance toward targets; sell legs first (same as backtester)."""
    self.cancel_all_orders()
    pos = self.get_pos()
    legs = []
    for ticker, target_pos in target.items():
        if ticker == "USD":
            continue
        delta = target_pos - pos.get(ticker, 0.0)
        if abs(delta) < 1e-6:
            continue
        legs.append((ticker, abs(delta), "buy" if delta > 0 else "sell"))
    legs.sort(key=lambda x: (x[2] == "buy", x[0]))
    for ticker, qty, side in legs:
        self.create_order(ticker, qty, side)

Returning to the point about the 0.01 threshold, we use it because limit orders do not always fill. Hence, if we choose a higher threshold, we increase the risk of having many order not fill and thus the strategy being more unpredictable. Note that the fill assumptions are a large risk in this strategy. 

Part 5: Live Trading + Results 

Besides the above trading logic, logic was added to effectively rebalance the portfolio and track how much money was left in the account in order to not go into potential debt. 

In [ ]:
#Rebalance Logic
def trade_to_target(self, target: Dict[str, float]):
    self.cancel_all_orders()
    pos = self.get_pos()
    legs = []
    for ticker, target_pos in target.items():
        delta = target_pos - pos.get(ticker, 0.0)
        if abs(delta) > 1e-6:
            legs.append((ticker, abs(delta), "buy" if delta > 0 else "sell"))
    legs.sort(key=lambda x: (x[2] == "buy", x[0]))  # sells before buys
    for ticker, qty, side in legs:
        self.create_order(ticker, qty, side)

In [ ]:
#Cash Aware Sizing
def max_affordable_buy_notional(cash, fill_px, fee_coef=0.07, reserve=0.01) -> float:
    """Largest buy notional affordable including Kalshi fees."""
    budget = cash - reserve
    lo, hi = 0.0, budget
    for _ in range(48):
        mid = (lo + hi) * 0.5
        fee = estimate_fee_per_trade(mid / fill_px, fill_px, fee_coef)
        if mid + fee <= budget:
            lo = mid
        else:
            hi = mid
    return lo

Over 1 WCF game and 3 NBA finals games (so far), the strategy returned 33% on average per game, netting a total profit of $106. 

We report the average return per game.

Game 6 (WCF): 30% on 20$

Game 1 (NBA FINALS): 50% on 20$

Game 2 (NBA FINALS): 10% on 100$

Game 3 (NBA FINALS): 30% on 300$

Part 6: Conclusion

Overall, this project indicates that prediction markets such as Kalshi have cross-market momentum opportunities, especially in the realm of markets which have correlated positive and negative movements with respect to a finite time lag. That being said, the strategy suffers from drawbacks such as inconsistent fills on limit orders and last minute end of game trading, espeically on games when the series market could close due to one team winning a series, causing one side of the market to got to 0. The positive returns suggest that this strategy could be extended to other sporting events that have similarly correlated markets. For example, an increase or decrease in the probability of a team winning a world cup game could be correlated with a similar change in their odds in the market for the world cup winner. 